# Explaining limitations clearly

_Doing ML for Real — the skills that matter (2026)_

**A model that won't say "I don't know" is dangerous — quantify what it doesn't know and write it down.**

Honest ML rests on one shift: stop emitting a single answer, start emitting an answer plus how much to trust it.

       Two ideas do the heavy lifting. Conformal prediction turns any model into one that outputs a set with a hard coverage guarantee — "the truth is in here at least 90% of the time." Calibration checks that the model's stated probabilities match real-world frequencies.

---

This notebook is a practice scaffold for the **Explaining limitations clearly** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q mapie
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — MAPIE + scikit-learn

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import calibration_curve
from mapie.classification import MapieClassifier

X, y = load_breast_cancer(return_X_y=True)
# train / calibration / test  -- calibration must be unseen by the model
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.4, random_state=0)
X_cal, X_te, y_cal, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=0)

clf = GradientBoostingClassifier(random_state=0).fit(X_tr, y_tr)

# 1) CONFORMAL PREDICTION SETS at target coverage 1 - alpha = 0.90
#    cv="prefit" tells MAPIE the estimator is already fit; it only calibrates.
mapie = MapieClassifier(estimator=clf, method="score", cv="prefit")
mapie.fit(X_cal, y_cal)
_, y_sets = mapie.predict(X_te, alpha=0.10)        # y_sets shape: (n_test, n_classes, 1)

covered = y_sets[np.arange(len(y_te)), y_te, 0]     # was the true label in the set?
print("target coverage : 0.90")
print("achieved coverage:", round(covered.mean(), 3))
print("avg set size     :", round(y_sets[:, :, 0].sum(axis=1).mean(), 3))  # >1 => abstain

# 2) CALIBRATION: are predicted 0.7s right ~70% of the time?
p_pos = clf.predict_proba(X_te)[:, 1]
frac_pos, mean_pred = calibration_curve(y_te, p_pos, n_bins=10, strategy="uniform")
ece = np.average(np.abs(frac_pos - mean_pred),
                 weights=np.histogram(p_pos, bins=10, range=(0, 1))[0][
                     np.unique(np.clip((p_pos * 10).astype(int), 0, 9))] if False else None)
# simpler, robust ECE over the returned bins:
ece = float(np.mean(np.abs(frac_pos - mean_pred)))
print("ECE (lower=better):", round(ece, 3))

# 3) ABSTENTION rule: defer to a human when the set is not a single confident label
defer = y_sets[:, :, 0].sum(axis=1) != 1
print("abstain / defer rate:", round(defer.mean(), 3))

## Visualize the data & results

_When the model says it's X% sure, is it right X% of the time?_

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)

clf = LogisticRegression(max_iter=5000).fit(X_tr, y_tr)
p = clf.predict_proba(X_te)[:, 1]

# observed frequency vs mean predicted prob, per bin -> the reliability curve
frac_pos, mean_pred = calibration_curve(y_te, p, n_bins=7, strategy="uniform")
for mp, fp in zip(mean_pred, frac_pos):
    print(round(float(mp), 2), round(float(fp), 2))   # (x, y) of the blue points

# Expected Calibration Error over the populated bins
ece = float(np.mean(np.abs(frac_pos - mean_pred)))
print("ECE:", round(ece, 3))   # ~0.03

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A teammate ships a churn model: "AUC 0.91, ready to go." A reviewer asks, "Is it ready to make retention-offer decisions?" What is missing before anyone acts on it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Ask for the SCOPE behind the 0.91 — which customers, what time window, what data sources was it validated on? — _A metric with no scope is a marketing claim; the model may be 0.91 on last year's data and far worse on a new segment._
- Demand a SUBGROUP breakdown of AUC by tenure, plan tier, and region. — _An aggregate AUC can hide a subgroup where the model is near-random, and retention offers would systematically misfire there._
- Check CALIBRATION — is a predicted 0.8 churn risk really ~80%? — _Offer budgets are spent on the probability; an uncalibrated score wastes money on the wrong customers._
- Ask for an ABSTENTION rule for borderline scores. — _Near the decision boundary the model adds little; deferring those to a human or to a hold-out test is cheaper than a wrong offer._

**Answer:** "AUC 0.91" is necessary but nowhere near sufficient. Before action you need: the validated scope, subgroup AUC (especially the worst group), a calibration check on the probabilities the offers depend on, and an abstention path for borderline cases. A one-page model card capturing intended use, scope, and known failure modes is the deliverable that makes it "ready."

</details>

**Problem 2.** You have a fitted classifier and want 90% coverage prediction sets. Sketch the split-conformal procedure and how you'd use MAPIE.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Hold out a fresh CALIBRATION split, separate from train and test. — _The coverage guarantee relies on calibration scores being exchangeable with test scores — they must not have been used to fit the model._
- Compute nonconformity scores $s_i = 1-\hat p(\text{true class})$ on the calibration split. — _Large scores mean the model was surprised by the truth; their distribution sets the threshold._
- Take $\hat q$ at rank $\lceil (n+1)(0.9)\rceil$, then include every label with score $\le \hat q$. — _The $(n+1)$ correction is what makes the finite-sample $\ge 0.9$ guarantee exact._
- In code: fit a base estimator, wrap it in MapieClassifier with method="score", call fit on the calibration data, then predict(..., alpha=0.1). — _MAPIE automates the score, the quantile, and the set construction; you supply $\alpha$._

**Answer:** Split-conformal: separate calibration data, score nonconformity $s_i=1-\hat p(\text{true})$, take the quantile $\hat q$ at rank $\lceil (n+1)(1-\alpha)\rceil$, and output the set of labels scoring $\le \hat q$. With MAPIE: MapieClassifier(estimator, method="score", cv="prefit"), .fit(X_cal, y_cal), then .predict(X_test, alpha=0.1) returns the prediction sets. Verify the achieved coverage on test data matches the 0.9 target.

</details>

**Problem 3.** A reliability diagram shows your model's curve sagging below the diagonal — at predicted 0.8 the observed frequency is only 0.6. What does this mean and what do you do?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read the gap: predicted > observed means the model is OVER-confident. — _It claims 0.8 but is right 0.6 of the time; decisions thresholded on probability will fire too eagerly._
- Quantify it with ECE (Expected Calibration Error) across bins. — _A single number lets you track calibration over time and compare before/after fixing it._
- Recalibrate with isotonic regression or Platt scaling on a held-out set. — _These monotonic maps pull the stated probabilities back onto the diagonal without retraining the base model._
- Re-plot the diagram and recompute ECE to confirm the fix. — _Calibration is verified empirically, never assumed._

**Answer:** The model is over-confident: its 0.8 predictions are right only ~60% of the time. Summarize with ECE, then recalibrate with isotonic regression or Platt scaling on held-out data, and re-plot to confirm the curve sits on the diagonal. Until then, do not present those probabilities as confidences — and use calibration_curve to monitor it going forward.

</details>